# Separating global and local components with principal component spatial filtering

fNIRS signals mix haemodynamic changes in the cortex with **systemic** changes:
blood pressure, respiration, Mayer waves and changes in scalp blood flow. Systemic
changes affect the whole head, so they show up in nearly every channel. They are
often also **correlated with the task**, for example when a block design raises heart
rate or blood pressure. Temporal methods (band-pass filters, regressing out a template)
then cannot separate them from the brain response, because the two have similar time
courses.

<cite data-cite="Zhang2016">Zhang, Noah and Hirsch (2016)</cite> use a different
property: **spatial extent**. Cortical activation is focal (a few centimetres), while
systemic signals are spread over the whole probe. Their *principal component spatial
filter* splits the measured signal into

- a **global component**: what remains after strongly smoothing the data in space, and
- a **local component**: the measured signal minus the global component, which keeps
  the focal, task-related activation.

Unlike the earlier eigenvector-based filter of
<cite data-cite="Zhang2005">Zhang et al. (2005)</cite>, it needs no resting-state
recording and does not remove whole principal components, which can also contain
neural signal.

This notebook explains the method, shows how to run it with Cedalion on a
high-density finger-tapping recording, and shows how the kernel width and the probe
coverage affect the result.

**Functions used**

| Function | Purpose |
|---|---|
| `cedalion.sigproc.physio.get_spatial_smoothing_kernel` | Gaussian smoothing kernel $K$ from 3D positions |
| `cedalion.sigproc.physio.compute_Hglobal_from_PCA` | global component $H_\mathrm{global}$ |
| `cedalion.models.glm.design_matrix.spatially_smoothed_pca_global_component_regressor` | the global component as a GLM regressor |

## The method

Let $H$ be the $n \times p$ data matrix of one chromophore, with $n$ time samples
(rows) and $p$ channels or vertices (columns). The singular value decomposition
splits it into temporal and spatial parts:

$$H = U \Sigma V^T$$

Each column of $U$ is the time course of one principal component (PC), each column
$v_i$ of $V$ is its spatial pattern, and $\Sigma$ holds the component strengths.

**Spatial smoothing.** Every spatial pattern is smoothed with a wide Gaussian kernel
$K$. This removes localized structure from the pattern and keeps only structure that
extends over large distances:

$$v_i^* = K v_i, \qquad V^* = K V$$

**Reconstruction.** The global component is rebuilt from the original time courses
and the smoothed patterns, and the local component is what is left:

$$H_\mathrm{global} = U \Sigma (V^*)^T, \qquad H_\mathrm{local} = H - H_\mathrm{global}$$

The kernel width $\sigma$ must be much larger than the expected activation but smaller
than the probe coverage. The paper uses $\sigma = 50°$ of arc on a sphere centred in
the brain, roughly ten times the 8 mm kernel common in fMRI.

### A useful way to read this

Because all components are kept, the decomposition cancels out:
$U \Sigma V^T K^T = H K^T$. The global component is therefore simply the data
**spatially low-pass filtered** at every time sample, and the local component is the
data **spatially high-pass filtered**. The PCA view explains where the method comes
from; the filter view makes its behaviour easier to predict. Like any spatial
high-pass filter, it attenuates activation that is broad compared with $\sigma$ and
can create negative "surround" activity next to a focal activation. We verify this
equivalence numerically below.

### Implementation notes

Cedalion's implementation differs from the paper in a few details:

| | Paper | Cedalion |
|---|---|---|
| Distance between positions | arc length on a sphere | straight-line (Euclidean) distance |
| Kernel | $\exp(-r^2/2\sigma^2)$, $\sigma = 50°$ | $\exp(-d^2/\sigma^2)$, `sigma_mm=80` by default, i.e. a Gaussian with standard deviation $\sigma/\sqrt{2} \approx 57$ mm |
| Normalisation | not specified | rows of $K$ sum to 1, so a spatially constant signal passes through unchanged; weights $< 10^{-3}$ are set to 0 |
| Centring | none | the spatial mean of each time sample is removed before the SVD and added back afterwards |
| Spatial domain | channels in MNI space | primarily vertices of an image reconstruction; channel space works but triggers a warning |

### When to use it, and when not

The paper and its simulations point out several limitations:

- **The probe must cover a much larger area than the activation.** In the simulations
  of <cite data-cite="Zhang2016">Zhang et al. (2016)</cite>, a 1 cm activation was
  underestimated and surrounded by artificial negative activity when the kernel was
  narrower than about 3 cm or the coverage smaller than about 9 cm.
- **Localized artefacts stay in the local component.** Signals from a large superficial
  vessel under a single optode, for example, are not removed.
- **Widespread brain activation is treated as global.** Tasks that activate large parts
  of the covered cortex, or several regions at once, lose part of their response.

Short-separation regression (see the GLM examples) uses a measured estimate of the
superficial signal instead of a spatial assumption. The two approaches have different
assumptions and can complement each other.

In [ ]:
# This cell sets up the environment when executed in Google Colab.
try:
    import google.colab
    !curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py
    # Select branch with --branch "branch name" (default is "dev")
    %run colab_setup.py
except ImportError:
    pass

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import cedalion
import cedalion.data
import cedalion.models.glm as glm
import cedalion.nirs
import cedalion.sigproc.frequency
import cedalion.vis.anatomy
from cedalion import units
from cedalion.sigproc.physio import (
    compute_Hglobal_from_PCA,
    get_spatial_smoothing_kernel,
)

xr.set_options(display_expand_data=False);

## Load and preprocess the data

We use the high-density finger-tapping dataset from the IBS Lab (100 channels over
both sensorimotor cortices, about 17 cm across). Participants tapped with the left or
right hand, or squeezed a ball. We convert the raw intensities to HbO/HbR concentration
changes and band-pass filter them between 0.01 and 0.5 Hz to remove slow drifts and the
cardiac pulsation.

In [ ]:
rec = cedalion.data.get_fingertappingDOT()

rec.stim.cd.rename_events(
    {
        "1": "Control",
        "2": "FTapping/Left",
        "3": "FTapping/Right",
        "4": "BallSqueezing/Left",
        "5": "BallSqueezing/Right",
    }
)

dpf = xr.DataArray(
    [6, 6], dims="wavelength", coords={"wavelength": rec["amp"].wavelength}
)
rec["od"] = cedalion.nirs.cw.int2od(rec["amp"])
rec["conc"] = cedalion.nirs.cw.od2conc(rec["od"], rec.geo3d, dpf, spectrum="prahl")
rec["conc"] = cedalion.sigproc.frequency.freq_filter(
    rec["conc"], 0.01 * units.Hz, 0.5 * units.Hz
)

ts = rec["conc"]
ts

## Build the smoothing kernel

The kernel needs a 3D position for each column of $H$. In channel space we use the
midpoint between source and detector. The positions must be in millimetres;
`get_spatial_smoothing_kernel` converts quantified DataArrays automatically.

With image-reconstructed data (`spatial_dim="vertex"`), pass the vertex coordinates of
the brain surface instead, for example `head_model.brain.vertices`.

In [ ]:
geo3d = rec.geo3d.pint.to("mm")
channel_pos = (geo3d.loc[ts.source] + geo3d.loc[ts.detector]) / 2

sigma_mm = 80.0
K = get_spatial_smoothing_kernel(channel_pos, sigma_mm=sigma_mm)

print(f"kernel shape: {K.shape}")
print(f"row sums: min {K.sum(1).min():.3f}, max {K.sum(1).max():.3f}")

Each row of $K$ holds the weights used to compute the smoothed value at one channel.
The left panel shows these weights for a single channel on the probe. The right panel
shows the weights of all channel pairs against their distance; they follow the
Gaussian $\exp(-d^2/\sigma^2)$, scaled by the row normalisation.

In [ ]:
from scipy.spatial.distance import cdist

i_ref = int(np.argmin(channel_pos.pint.dequantify().values[:, 0]))  # leftmost channel
ref_ch = ts.channel.values[i_ref]
dist = cdist(channel_pos.pint.dequantify().values, channel_pos.pint.dequantify().values)

f, ax = plt.subplots(1, 2, figsize=(12, 5))
cedalion.vis.anatomy.scalp_plot(
    ts,
    rec.geo3d,
    K[i_ref],
    ax[0],
    cmap="viridis",
    vmin=0,
    cb_label="kernel weight",
    title=f"weights for channel {ref_ch}",
)
ax[1].plot(dist.ravel(), K.ravel(), ".", ms=2, alpha=0.5)
d = np.linspace(0, dist.max(), 200)
ax[1].plot(d, np.exp(-(d**2) / sigma_mm**2) * K.max(), "k--", label=r"$\exp(-d^2/\sigma^2)$")
ax[1].set_xlabel("distance between channel midpoints / mm")
ax[1].set_ylabel("kernel weight")
ax[1].legend()
plt.tight_layout()

## Compute the global and local components

`compute_Hglobal_from_PCA` processes each chromophore separately. It warns because
the method is primarily intended for vertex space; here we use channel space on purpose,
so we suppress the warning.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    ts_global = compute_Hglobal_from_PCA(
        ts, smoothing_kernel=K, spatial_dim="channel", spectral_dim="chromo"
    )

ts_local = ts - ts_global
ts_global

Now we check the equivalence described above: the global component equals the data
multiplied by $K^T$ at every time sample. The remaining difference comes from the
single-precision (`float32`) arithmetic used inside the function.

In [ ]:
K_da = xr.DataArray(
    K,
    dims=("channel_out", "channel"),
    coords={"channel_out": ts.channel.values, "channel": ts.channel.values},
)
ts_smoothed = xr.dot(K_da, ts.pint.dequantify(), dim="channel").rename(
    channel_out="channel"
)

max_diff = abs(ts_global.pint.dequantify() - ts_smoothed).max().item()
print(f"max |H_global - H K^T| = {max_diff:.2e} µM")
print(f"max |H|              = {abs(ts).max().item():.2e}")

## Time courses

By construction, the global component varies only slowly across the probe, so nearby
channels share almost the same global time course. The local component keeps the
differences between channels.

In [ ]:
channels = ts.channel.values[[0, 25, 50, 75]]
tmin, tmax = 300, 500

f, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True, sharey=True)
for a, data, title in zip(
    ax,
    [ts, ts_global, ts_local],
    ["measured signal $H$", r"global component $H_\mathrm{global}$",
     r"local component $H_\mathrm{local} = H - H_\mathrm{global}$"],
):
    for ch in channels:
        a.plot(
            data.time,
            data.sel(channel=ch, chromo="HbO").pint.dequantify(),
            lw=1,
            label=ch,
        )
    a.set_title(f"{title} (HbO)")
    a.set_ylabel(r"$\Delta$c / µM")
ax[0].legend(ncol=4, loc="upper right")
ax[-1].set_xlabel("time / s")
ax[-1].set_xlim(tmin, tmax)
plt.tight_layout()

## Block averages

We now look at the response to right-hand finger tapping, which should activate the
left motor cortex. We cut the three signals into epochs from 5 s before to 25 s after
each stimulus onset, subtract the pre-stimulus baseline and average over trials.

In [ ]:
def block_average(data, trial_type="FTapping/Right"):
    epochs = data.cd.to_epochs(
        rec.stim, [trial_type], before=5 * units.s, after=25 * units.s
    )
    baseline = epochs.sel(reltime=slice(None, 0)).mean("reltime")
    return (epochs - baseline).mean("epoch")


ba = {
    "measured": block_average(ts),
    "global": block_average(ts_global),
    "local": block_average(ts_local),
}

f, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for a, (name, avg) in zip(ax, ba.items()):
    for chromo, color in [("HbO", "r"), ("HbR", "b")]:
        a.plot(
            avg.reltime,
            avg.sel(chromo=chromo).pint.dequantify().T,
            color=color,
            lw=0.5,
            alpha=0.4,
        )
    a.axvspan(0, 10, color="0.9", zorder=-1)
    a.set_title(f"{name}: all channels (HbO red, HbR blue)")
    a.set_xlabel("time relative to onset / s")
ax[0].set_ylabel(r"$\Delta$c / µM")
plt.tight_layout()

The global component is not flat: its HbO time course rises during and after the
task in both hemispheres, so it is **locked to the task**. Temporal methods (filters,
regressing out a template) cannot separate such a signal from the brain response. The
global HbR component is much smaller than the global HbO component, which matches the
observation of <cite data-cite="Zhang2016">Zhang et al. (2016)</cite> that the global
component is mainly an HbO effect. The two bundles of global traces correspond to the
two probe patches: they lie about 13 cm apart, so the 80 mm kernel smooths mostly
within each patch.

The local component keeps the differences between channels, with responses of both
signs.

The scalp maps below show the mean concentration change between 5 and 15 s after onset
for each channel. Note that this example uses a single participant without channel
pruning or motion correction, so the maps are noisy; it shows how to apply the method,
not a validation of it. The simulation in the next section uses data where the true
activation is known.

In [ ]:
def response_map(avg):
    return avg.sel(reltime=slice(5, 15)).mean("reltime").pint.dequantify()


f, ax = plt.subplots(2, 3, figsize=(15, 9))
for i_chromo, chromo in enumerate(["HbO", "HbR"]):
    maps = {name: response_map(avg).sel(chromo=chromo) for name, avg in ba.items()}
    vmax = max(float(abs(m).max()) for m in maps.values())
    for i_name, (name, m) in enumerate(maps.items()):
        cedalion.vis.anatomy.scalp_plot(
            ts,
            rec.geo3d,
            m,
            ax[i_chromo, i_name],
            cmap="RdBu_r",
            vmin=-vmax,
            vmax=vmax,
            title=f"{chromo} {name}",
            cb_label=r"$\Delta$c / µM",
        )
plt.tight_layout()

## Simulation: choosing the kernel width

To see what the filter does when the truth is known, we simulate data on the channel
positions of this probe, similar to Fig. 5 of
<cite data-cite="Zhang2016">Zhang et al. (2016)</cite>:

- a **focal activation**: a Gaussian patch (standard deviation 10 mm) centred on one
  channel over the left hemisphere, following a block-design time course;
- a **global signal** that is the same in every channel. It contains the *same* task
  time course plus a 0.1 Hz Mayer-wave-like oscillation, so it is fully correlated with
  the task;
- white measurement noise.

For each channel, we estimate the activation amplitude by regressing its time course
on the task time course. We do this for the simulated measurement and for the local
component obtained with different kernel widths.

In [ ]:
rng = np.random.default_rng(42)
pos = channel_pos.pint.dequantify().values
fs = 5.0
t = np.arange(0, 600, 1 / fs)

# block design: 10 s task, 20 s rest, smoothed with a gamma-shaped HRF
boxcar = ((t % 30) < 10).astype(float)
t_hrf = np.arange(0, 20, 1 / fs)
hrf = t_hrf**5 * np.exp(-t_hrf)
task = np.convolve(boxcar, hrf / hrf.sum())[: t.size]
task -= task.mean()
task /= task.max()

# focal activation around a left-hemisphere channel
left = np.where(pos[:, 0] < 0)[0]
i_center = left[np.argmin(np.linalg.norm(pos[left] - pos[left].mean(0), axis=1))]
d_center = np.linalg.norm(pos - pos[i_center], axis=1)
true_map = np.exp(-(d_center**2) / (2 * 10.0**2))

global_tc = 1.0 * task + 0.5 * np.sin(2 * np.pi * 0.1 * t)
sim = (
    task[:, None] * true_map[None, :]
    + global_tc[:, None]
    + 0.1 * rng.standard_normal((t.size, pos.shape[0]))
)

ts_sim = xr.DataArray(
    sim[:, :, None],
    dims=("time", "channel", "chromo"),
    coords={"time": t, "channel": ts.channel.values, "chromo": ["HbO"]},
).pint.quantify("uM")


def activation_map(data):
    # least-squares amplitude of the task time course in every channel
    x = data.sel(chromo="HbO").pint.dequantify().values
    return (task @ x) / (task @ task)


maps = {"true activation": true_map, "simulated measurement": activation_map(ts_sim)}
for sigma in [20.0, 40.0, 80.0, 160.0]:
    K_s = get_spatial_smoothing_kernel(channel_pos, sigma_mm=sigma)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        glob_s = compute_Hglobal_from_PCA(
            ts_sim, smoothing_kernel=K_s, spatial_dim="channel", spectral_dim="chromo"
        )
    maps[f"local, sigma_mm = {sigma:.0f}"] = activation_map(ts_sim - glob_s)

f, ax = plt.subplots(2, 3, figsize=(15, 9))
for a, (title, m) in zip(ax.ravel(), maps.items()):
    cedalion.vis.anatomy.scalp_plot(
        ts, rec.geo3d, m, a, cmap="RdBu_r", vmin=-1.2, vmax=1.2,
        title=title, cb_label="amplitude",
    )
plt.tight_layout()

for title, m in maps.items():
    print(
        f"{title:>24}: peak {m[i_center]:+.2f}, min {m.min():+.2f}, "
        f"mean over right hemisphere {m[pos[:, 0] > 0].mean():+.2f}"
    )

The simulated measurement shows an amplitude of about 1 in *every* channel: the
task-locked global signal looks like activation everywhere, and only the focal patch
stands out on top of it.

The local component removes this offset, and the kernel width decides how well:

- With a **narrow kernel** (20 mm), which is only twice as wide as the activation, a
  third of the activation is counted as global: the peak drops from 1 to about 0.66.
- With **wider kernels** the peak is recovered better (about 0.82, 0.87 and 0.91 for
  40, 80 and 160 mm).
- For all widths, channels next to the activation become **negative** (down to about
  -0.1 to -0.17), although the true activation is zero there. This is the artificial
  "negative surround" of a spatial high-pass filter described in the paper. Negative
  values in the local component close to a strong activation should therefore be
  interpreted with care.
- With a kernel **as wide as the probe** (160 mm) the global component approaches the
  spatial mean of all channels, i.e. plain global-mean subtraction. The other
  hemisphere then gets a small negative offset, because the mean includes the
  activation itself.

The global offset of about 1 is removed in all cases, even though it has exactly the
same time course as the activation.

Choose $\sigma$ from the expected extent of the activation and the probe coverage
*before* looking at the results, as the authors did, to avoid tuning the analysis to the
outcome.

## Using the global component in a GLM

Instead of subtracting the global component before the analysis, it can be modelled
in the GLM.
`glm.design_matrix.spatially_smoothed_pca_global_component_regressor` computes the
global component as above, averages it over all channels and adds the resulting time
course (one per chromophore) as a common regressor to the design matrix. The GLM then
estimates how strongly each channel loads on it.

This is a different model from the subtraction above: subtraction removes a different
global time course from each channel (the smoothed field varies slowly across the
probe), while the regressor uses a single time course for all channels with a fitted
weight per channel.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    dm_global = glm.design_matrix.spatially_smoothed_pca_global_component_regressor(
        ts, channel_pos, sigma_mm=80.0, spatial_dim="channel", spectral_dim="chromo"
    )

dms = (
    glm.design_matrix.hrf_regressors(
        ts, rec.stim, glm.Gamma(tau=0 * units.s, sigma=3 * units.s)
    )
    & glm.design_matrix.drift_regressors(ts, drift_order=1)
)
dms_with_global = dms & dm_global

dms_with_global.common

The combined design matrix is fitted as usual, e.g. with
`glm.fit(ts, dms_with_global, noise_model="ols")`; see the GLM examples for fitting
and visualising the results.

## Summary

- The principal component spatial filter separates fNIRS signals by spatial extent:
  widespread (global) structure goes into $H_\mathrm{global}$, focal structure stays in
  $H_\mathrm{local} = H - H_\mathrm{global}$.
- With all components kept, this is the same as spatially low-pass filtering the data
  with the kernel $K$; $H_\mathrm{local}$ is a spatially high-pass filtered signal.
- It needs no resting-state data and no short-separation channels, and it also removes
  systemic signals that are correlated with the task.
- It requires a probe that covers a much larger area than the activation, and a kernel
  that is much wider than the activation. Otherwise, responses are underestimated and
  artificial negative surround activity appears.
- It works best on vertex-space data after image reconstruction, where vertex positions
  describe the cortical geometry better than channel midpoints.

## References

- <cite data-cite="Zhang2016">X. Zhang, J. A. Noah, J. Hirsch (2016). Separation of the
  global and local components in functional near-infrared spectroscopy signals using
  principal component spatial filtering. *Neurophotonics* 3(1), 015004.</cite>
- <cite data-cite="Zhang2005">Y. Zhang, D. H. Brooks, M. A. Franceschini, D. A. Boas
  (2005). Eigenvector-based spatial filtering for reduction of physiological
  interference in diffuse optical imaging. *J. Biomed. Opt.* 10(1), 011014.</cite>